In [1]:
# Core imports for the whole lab
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)
# A SMALL, ALREADY-CLEAN DATASET (Part 1 did the cleaning)
# Mixed columns: an unordered category (city), an ordered category
# (size), two numeric features on very different scales, and a target.
df = pd.DataFrame({
    'city':   ['pune', 'delhi', 'mumbai', 'pune', 'delhi', 'mumbai', 'pune', 'delhi'],
    'size':   ['small', 'large', 'medium', 'medium', 'small', 'large', 'large', 'small'],
    'age':    [25, 41, 33, 29, 52, 38, 46, 22],
    'income': [38000, 92000, 55000, 47000, 120000, 76000, 88000, 41000],
    'bought': [0, 1, 0, 0, 1, 1, 1, 0],   # target
})
df

Setup complete. pandas 2.2.3


,city,size,age,income,bought
0,pune,small,25,38000,0
1,delhi,large,41,92000,1
2,mumbai,medium,33,55000,0
3,pune,medium,29,47000,0
4,delhi,small,52,120000,1
5,mumbai,large,38,76000,1
6,pune,large,46,88000,1
7,delhi,small,22,41000,0


In [2]:
#1.Encoding categorical data
#  1A. ONE-HOT ENCODING (for UNORDERED categories)
# 'city' has no natural order -> one 0/1 column per category
city_ohe = pd.get_dummies(df['city'], prefix='city').astype(int)
city_ohe
# 1B. LABEL / ORDINAL ENCODING (for ORDERED categories)
# 'size' has a real order: small < medium < large -> map to 0,1,2
size_order = {'small': 0, 'medium': 1, 'large': 2}
df['size_code'] = df['size'].map(size_order)
df[['size', 'size_code']]

,size,size_code
0,small,0
1,large,2
2,medium,1
3,medium,1
4,small,0
5,large,2
6,large,2
7,small,0


In [6]:
#LAB EXERCISE-1
# 1. One-hot encode city if it has not already been encoded
if 'city' in df.columns:
    df = pd.get_dummies(df, columns=['city'], dtype=int)

# 2. Ordinal-encode size
size_order = {'small': 0, 'medium': 1, 'large': 2}
if 'size' in df.columns:
    df['size_code'] = df['size'].map(size_order)

# 3. Explanation:
# One-hot encoding is not ideal for size because it has a natural order.
# Ordinal encoding is not ideal for city because cities have no natural order.

# Display the result
display(df.head())

,size,age,income,bought,size_code,city_delhi,city_mumbai,city_pune
0,small,25,38000,0,0,0,0,1
1,large,41,92000,1,2,1,0,0
2,medium,33,55000,0,1,0,1,0
3,medium,29,47000,0,1,0,0,1
4,small,52,120000,1,0,1,0,0


In [8]:
#2.Feature scaling
# 2A. THE PROBLEM — FEATURES ON DIFFERENT SCALES
# income (tens of thousands) dwarfs age (tens). A distance-based
# model would treat income as nearly all that matters.
print(df[['age', 'income']].describe().loc[['min', 'max', 'mean']])
# 2B. STANDARDISATION (Z-score)  vs  NORMALISATION (Min-Max)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
num = df[['age', 'income']]
z = StandardScaler().fit_transform(num)        # mean 0, std 1
m = MinMaxScaler().fit_transform(num)          # range [0, 1]
print('Standardised (mean~0, std~1):')
print(pd.DataFrame(z, columns=['age', 'income']).round(2).head(3))
print('\nMin-Max (range 0..1):')
print(pd.DataFrame(m, columns=['age', 'income']).round(2).head(3))


        age    income
min   22.00   38000.0
max   52.00  120000.0
mean  35.75   69625.0
Standardised (mean~0, std~1):
    age  income
0 -1.10   -1.16
1  0.54    0.82
2 -0.28   -0.54

Min-Max (range 0..1):
    age  income
0  0.10    0.00
1  0.63    0.66
2  0.37    0.21


In [9]:
#LAB EXERCISE-2
from sklearn.preprocessing import StandardScaler, MinMaxScaler
income = df[['income']]  # 2D shape for sklearn
# 1. Standardise -> check mean ~0, std ~1
z = StandardScaler().fit_transform(income)
print("Standardised income:")
print(pd.DataFrame(z, columns=['income']).head())
print("Mean:", z.mean().round(6))
print("Standard deviation:", z.std().round(6))

# 2. Min-Max scale -> check min 0, max 1
m = MinMaxScaler().fit_transform(income)
print("\nMin-Max scaled income:")
print(pd.DataFrame(m, columns=['income']).head())
print("Minimum:", m.min().round(6))
print("Maximum:", m.max().round(6))

# 3. Model types:
# Needs scaling: K-Nearest Neighbors (KNN).
# Doesn't usually need scaling: Decision Tree.

Standardised income:
     income
0 -1.158468
1  0.819628
2 -0.535734
3 -0.828786
4  1.845307
Mean: -0.0
Standard deviation: 1.0

Min-Max scaled income:
     income
0  0.000000
1  0.658537
2  0.207317
3  0.109756
4  1.000000
Minimum: 0.0
Maximum: 1.0


In [10]:
#3. Avoiding data leakage
# 3A. SPLIT FIRST, THEN FIT ON TRAIN ONLY
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X = df[['age', 'income']]
y = df['bought']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # FIT + transform on train
X_test_s  = scaler.transform(X_test)        # only TRANSFORM on test
print('train rows:', X_train.shape[0], '| test rows:', X_test.shape[0])
print('scaler learned mean from TRAIN only:', scaler.mean_.round(1))

train rows: 6 | test rows: 2
scaler learned mean from TRAIN only: [3.45e+01 6.90e+04]


In [11]:
# LAB EXERCISE 3 — Spot the leak
from sklearn.preprocessing import StandardScaler
# 1a. WRONG: fit on ALL data, then split (leakage!)
wrong_mean = StandardScaler().fit(X).mean_

# 1b. RIGHT: fit on TRAIN only
right_mean = StandardScaler().fit(X_train).mean_

print('fit-on-all mean :', wrong_mean.round(1))
print('fit-on-train mean:', right_mean.round(1))

# 2. Why fitting on all data is a problem:
# It causes data leakage because the scaler learns information from the
# test set. This can make model evaluation unreliable. Fit the scaler
# on training data only, then use transform() on the test data.

fit-on-all mean : [3.5800e+01 6.9625e+04]
fit-on-train mean: [3.45e+01 6.90e+04]


In [12]:
#4. Feature engineering
#  4A. COMBINE & BIN EXISTING COLUMNS
fe = df.copy()
# combine: income per year of age (a crude 'earning rate')
fe['income_per_age'] = (fe['income'] / fe['age']).round(0)
# bin: turn continuous age into life-stage buckets
fe['age_group'] = pd.cut(fe['age'], bins=[0, 30, 45, 100],
                       labels=['young', 'mid', 'senior'])
fe[['age', 'income', 'income_per_age', 'age_group']]
# 4B. EXTRACT FROM A DATE
dates = pd.to_datetime(['2024-01-06', '2024-03-15', '2024-07-21', '2024-12-25'])
d = pd.DataFrame({'date': dates})
d['day_of_week'] = d['date'].dt.day_name()   # Monday, Tuesday, ...
d['month']       = d['date'].dt.month
d['is_weekend']  = d['date'].dt.dayofweek >= 5
d

,date,day_of_week,month,is_weekend
0,2024-01-06,Saturday,1,True
1,2024-03-15,Friday,3,False
2,2024-07-21,Sunday,7,True
3,2024-12-25,Wednesday,12,False


In [13]:
# LAB EXERCISE 4 — Engineer two features
ex = df.copy()
# 1. Create high_earner flag (income > median)
ex['high_earner'] = ex['income'] > ex['income'].median()

# 2. Bin income into 3 labelled buckets
ex['income_group'] = pd.cut(
    ex['income'],
    bins=3,
    labels=['low', 'mid', 'high'],
    include_lowest=True
)

# 3. Show income + the new columns
display(ex[['income', 'high_earner', 'income_group']])

,income,high_earner,income_group
0,38000,False,low
1,92000,True,mid
2,55000,False,low
3,47000,False,low
4,120000,True,high
5,76000,True,mid
6,88000,True,mid
7,41000,False,low


In [18]:
# 5. THE PREPROCESSING PIPELINE
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
# Restore original df if city column is missing
if 'city' not in df.columns:
    df = pd.DataFrame({
        'age': [25, 32, 47, 29, 41, 36, 52, 23, 38, 45, 31, 28],
        'income': [30000, 50000, 80000, 35000, 70000, 60000,
                   90000, 25000, 65000, 85000, 45000, 32000],
        'city': ['Delhi', 'Mumbai', 'Chennai', 'Delhi', 'Mumbai',
                 'Chennai', 'Delhi', 'Mumbai', 'Chennai', 'Delhi',
                 'Mumbai', 'Chennai'],
        'size': ['small', 'medium', 'large', 'small', 'large',
                 'medium', 'large', 'small', 'medium', 'large',
                 'medium', 'small'],
        'bought': [0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0]
    })
# Select features and target
num_cols = ['age', 'income']
cat_cols = ['city', 'size']
X = df[num_cols + cat_cols]
y = df['bought']
# Preprocessing
pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])
# Create pipeline
pipe = Pipeline([
    ('prep', pre),
    ('model', LogisticRegression(max_iter=1000))
])
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)
# Train and evaluate
pipe.fit(X_train, y_train)
acc = pipe.score(X_test, y_test)
print("Pipeline:", pipe)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Test accuracy:", round(acc, 2))

Pipeline: Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'income']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'size'])])),
                ('model', LogisticRegression(max_iter=1000))])
Training rows: 9
Testing rows: 3
Test accuracy: 1.0


In [21]:
#LAB EXERCISE-5
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
# 1. Select numeric features and target
Xn = df[['age', 'income']]
yn = df['bought']
# Create Pipeline: MinMaxScaler -> LogisticRegression
pipe = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

# 2. Split data and fit on training set
X_train, X_test, y_train, y_test = train_test_split(
    Xn, yn, test_size=0.25, random_state=0
)
pipe.fit(X_train, y_train)

# 3. Print test accuracy
accuracy = pipe.score(X_test, y_test)
print('Test accuracy:', round(accuracy, 2))

Test accuracy: 1.0
